# latex-task1 · Binding-affinity tables → LaTeX

Converts the affinity tables in `results/reports/results.html` (Table 1a / CL1-CL2 / similarity-stratified / CASF / CASF-clean).

## Setup

In [ ]:
import sys
from pathlib import Path
def _find_repo_root():
    for c in [Path.cwd(), *Path.cwd().parents]:
        if (c / "voxbind" / "dataset").is_dir():
            return c
    fb = Path("/home/shpark/prj-denovo/VoxBind")
    if (fb / "voxbind" / "dataset").is_dir():
        return fb
    raise FileNotFoundError("repo root (voxbind/dataset) not found from cwd")
_REPO = _find_repo_root()
sys.path.insert(0, str(_REPO / "notebook" / "results"))
from bs4 import BeautifulSoup, Tag
from latex_common import *   # shared HTML->LaTeX helpers + generic table_to_latex

# ── task1 config (binding-affinity report) ──
HTML_PATH = Path("results/reports/results.html")
KEEP_BADGES = True
TABLE1_EXCLUDE_TAGS = True
RESIZE_WIDE = True
CDG_VARIANT = "v2"   # which Ours(CDG) row to show: HTML key f"CDG {CDG_VARIANT}"
TABLE1_GROUP_HEADERS = [(2, "Test (N=1312)"), (11, "CL3 filtered (N=733)")]
TABLE1_CAPTION = (
    r"\textbf{LP-PDBBind binding-affinity prediction} on the original and CL3-filtered "
    r"test sets. \textbf{Bold} marks the best mean and results whose means fall within "
    r"each other's standard-deviation intervals; \underline{underlining} marks the next-best "
    r"mean outside that group.")
SIMILARITY_GROUP_HEADERS = [(8, "Similarity < 30% (N=453)"), (5, "Similarity < 60% (N=813)")]
SIMILARITY_EXCLUDE_TAGS = True
SIMILARITY_CAPTION = (
    r"\textbf{LP-PDBBind similarity-stratified binding-affinity prediction} on "
    r"protein-similarity-filtered test sets. \textbf{Bold} marks the best "
    r"mean and results whose means fall within each other's standard-deviation intervals; "
    r"\underline{underlining} marks the next-best mean outside that group.")


## Converters (affinity)

In [ ]:
BINDING_MODALITY_GROUPS = [
    ("Sequence", [
        ("DeepDTA",              "DeepDTA",           r"\xmark", False),
        ("MolTrans",             "MolTrans",          r"\xmark", False),
        ("HonestAffinity",       "HonestAffinity",    r"\cmark", False),
        (r"Nesso-1$^\dagger$",   "Nesso-1",           r"\cmark", True),
    ]),
    ("Structure", [
        ("HBGSA",                "HBGSA",             r"\xmark", False),
        ("EGNN",                 "EGNN supervised",   r"\xmark", False),
        ("EGNN + TargetDiff",    "EGNN + TargetDiff", r"\xmark", False),
        ("GET",                  "GET",               r"\xmark", False),
        ("CheapNet",             "CheapNet",          r"\xmark", False),
        ("IPNet",                "IPNet (scratch)",   r"\xmark", False),
        (r"IPNet$^\dagger$",     "IPNet (frozen)",    r"\cmark", True),
        ("AEV-PLIG",             "AEV-PLIG",          r"\xmark", False),
        ("DSMBind",              "DSMBind",           r"\cmark", False),
        ("BindNet",              "BindNet",           r"\cmark", False),
        ("ProFSA",               "ProFSA",            r"\cmark", False),
        (r"Ours (C)",             "C pretrained",      r"\cmark", False),
    ]),
    ("Density", [
        # (우선 LaTeX 제외) ("CDG (mask050)", "C+D+G pretrained", r"\cmark", False),
        # (우선 LaTeX 제외) ("CDG + corr", "C+D+G +corr", r"\cmark", False),
        # 여러 CDG 변형 중 CDG_VARIANT(예: v2) 하나만 "Ours (CDG)"로 표시.
        (r"Ours (CDG)",          f"CDG {CDG_VARIANT}", r"\cmark", False),
    ]),
]
# 이 표시명 앞에 \cmidrule(lr){2-9} (baseline vs ours 구분)
BINDING_CMIDRULE_BEFORE = {"Ours (C)"}
BINDING_GROUP_HEADERS = ("Test (N=1312)", "CL3 filtered Test (N=733)")
BINDING_CAPTION = (
    r"\textbf{LP-PDBBind binding-affinity prediction on the \textit{test} and "
    r"\textit{CL3-filtered test} sets.} Metrics are reported as the mean and standard "
    r"deviation over five seeds. \textbf{Bold} marks the best and \underline{underlining} "
    r"marks the next-best; methods whose intervals overlap are marked together. $\dagger$ "
    r"denotes methods trained on leaked data overlapping with the test set; these are "
    r"excluded from marking as they are not comparable to leakage-controlled methods. "
    r"N/A denotes unavailable metrics, and \textbf{Pre.} denotes whether models use "
    r"pre-training on external data."
)


SIMILARITY_MODALITY_CAPTION = (
    r"\textbf{LP-PDBBind similarity-stratified binding-affinity prediction on the "
    r"\textit{CL3-filtered test} set, split by maximum protein-sequence identity to the "
    r"CL3 training set.} Metrics are reported as the mean and standard deviation over five "
    r"seeds. \textbf{Bold} marks the best and \underline{underlining} marks the next-best; "
    r"methods whose intervals overlap are marked together. $\dagger$ denotes methods "
    r"trained on leaked data overlapping with the test set; these are excluded from marking "
    r"as they are not comparable to leakage-controlled methods. N/A denotes unavailable "
    r"metrics, and \textbf{Pre.} denotes whether models use pre-training on external data."
)


BINDING_CL1_CL2_CAPTION = (
    r"\textbf{LP-PDBBind binding-affinity prediction on the \textit{CL1-filtered} and "
    r"\textit{CL2-filtered} test sets.} Metrics are reported as the mean and standard "
    r"deviation over five seeds. \textbf{Bold} marks the best and \underline{underlining} "
    r"marks the next-best; methods whose intervals overlap are marked together. $\dagger$ "
    r"denotes methods trained on leaked data overlapping with the test set; these are "
    r"excluded from marking as they are not comparable to leakage-controlled methods. "
    r"N/A denotes unavailable metrics, and \textbf{Pre.} denotes whether models use "
    r"pre-training on external data."
)


def _binding_parse_metric(cell: Tag):
    tbd = cell.find(class_="tbd")
    if tbd is not None and tbd.get_text(strip=True).lower() in ("n/a", "na"):
        return (None, None)               # explicit not-applicable (zero-shot energy RMSE) → N/A
    val = cell.find(class_="val")
    if val is None:                       # TBA / 빈 셀 → 값 없음
        return None
    sd = cell.find(class_="sd")
    number = r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)"
    mean = float(re.search(number, val.get_text(strip=True).replace("−", "-")).group())
    std = abs(float(re.search(number, sd.get_text(strip=True)).group())) if sd else 0.0
    return mean, std


def _modality_grouped_to_latex(table, metric_indices, group_headers, caption, label, expect_metrics=12, casf_clean=frozenset()):
    # 공통 렌더러: modality 그룹 + Pre. 열 + dagger + bold/underline (binding / similarity 공용)
    html_rows = []
    for row in table.find_all("tr"):
        method_cell = row.find("td", class_="col-method", recursive=False)
        metrics = row.find_all("td", class_="metric", recursive=False)
        if method_cell is None or len(metrics) != expect_metrics:
            continue
        text = method_cell.get_text(" ", strip=True)
        vals = [_binding_parse_metric(metrics[i]) for i in metric_indices]
        html_rows.append((text, vals))

    resolved = []  # (표시명, pre, leaked, [6 (mean,std)])
    for _, methods in BINDING_MODALITY_GROUPS:
        for display, key, pre, leaked in methods:
            match = next((v for text, v in html_rows if text.startswith(key)), None)
            if match is None:
                raise ValueError(f"results.html에서 {key!r}로 시작하는 행을 찾지 못했습니다.")
            disp_e, leaked_e = ((display.replace(r"$^\dagger$", ""), False)
                               if key in casf_clean else (display, leaked))
            resolved.append((disp_e, pre, leaked_e, match))

    # 열별 강조: leaked 행은 순위에서 제외
    maximize = [True, True, False] * len(group_headers)  # r↑ ρ↑ RMSE↓ per group
    ncols_metric = len(maximize)
    bold = [set() for _ in range(ncols_metric)]
    under = [set() for _ in range(ncols_metric)]
    for col in range(ncols_metric):
        pts = [(i, r[3][col][0], r[3][col][1]) for i, r in enumerate(resolved)
               if not r[2] and r[3][col] is not None and r[3][col][0] is not None]
        if not pts:
            continue
        best = (max if maximize[col] else min)(m for _, m, _ in pts)
        best_std = max(s for _, m, s in pts if abs(m - best) < 1e-9)
        for i, m, s in pts:
            gap = abs(m - best)
            if gap <= best_std and gap <= s:
                bold[col].add(i)
        remaining = [(i, m) for i, m, s in pts if i not in bold[col]]
        if remaining:
            second = (max if maximize[col] else min)(m for _, m in remaining)
            for i, m in remaining:
                if abs(m - second) < 1e-9:
                    under[col].add(i)

    def cell_tex(i, col, pair):
        if pair is None:
            return "TBA"
        if pair[0] is None:               # explicit N/A cell (not-applicable, e.g. zero-shot energy RMSE)
            return "N/A"
        if not maximize[col] and pair[0] >= 100:   # off-scale RMSE (uncalibrated energy) → N/A
            return "N/A"
        text = rf"{pair[0]:.3f}\std{{{pair[1]:.3f}}}"
        if i in bold[col]:
            return rf"\textbf{{{text}}}"
        if i in under[col]:
            return rf"\underline{{{text}}}"
        return text

    lines = []
    i = 0
    for modality, methods in BINDING_MODALITY_GROUPS:
        lines.append(rf"\multirow{{{len(methods)}}}{{*}}{{{modality}}}")
        for display, key, pre, leaked in methods:
            disp_e, _, _, vals = resolved[i]
            if display in BINDING_CMIDRULE_BEFORE:
                lines.append(rf"\cmidrule(lr){{2-{ncols_metric + 3}}}")
            cells = [cell_tex(i, col, vals[col]) for col in range(ncols_metric)]
            lines.append("& " + " & ".join([disp_e, pre, *cells]) + r" \\")
            i += 1
        if modality != BINDING_MODALITY_GROUPS[-1][0]:
            lines.append(r"\midrule")

    header1 = (
        r"\multirow{3}{*}{\textbf{Input}} & \multirow{3}{*}{\textbf{Method}} & "
        r"\multirow{3}{*}{\textbf{Pre.}} & "
        + " & ".join(rf"\multicolumn{{3}}{{c}}{{\textbf{{{h}}}}}" for h in group_headers) + r" \\"
    )
    _mh = r"\textbf{Pearson \textit{r}} & \textbf{Spearman $\rho$} & \textbf{RMSE $\downarrow$}"
    header2 = r"& & & " + " & ".join([_mh] * len(group_headers)) + r" \\"
    body = [
        r"\begin{table}[!t]",
        r"    \centering",
        r"    \caption{",
        f"        {caption}",
        r"    }",
        rf"    \label{{{label}}}",
        r"    \resizebox{.98\textwidth}{!}{%",
        rf"        \begin{{tabular}}{{@{{}}ll{'c' * (ncols_metric + 1)}@{{}}}}",
        r"            \toprule",
        "            " + header1,
        "            " + "".join(rf"\cmidrule(lr){{{4 + 3 * g}-{6 + 3 * g}}}" for g in range(len(group_headers))),
        "            " + header2,
        r"            \midrule",
    ]
    body.extend("            " + ln for ln in lines)
    body.extend([
        r"            \bottomrule",
        r"        \end{tabular}",
        r"    }",
        r"\end{table}",
    ])
    return "\n".join(body)


def binding_affinity_to_latex(table: Tag) -> str:
    return _modality_grouped_to_latex(
        table, (0, 1, 2, 9, 10, 11), BINDING_GROUP_HEADERS,
        BINDING_CAPTION, "tab:result-binding-affinity", expect_metrics=12)


CASF_CAPTION = (
    r"\textbf{CASF-2016 external binding-affinity prediction.} All models are trained on "
    r"LP-PDBBind and evaluated on the held-out CASF-2016 core set, reported over two "
    r"cohorts. Metrics are reported as the mean and standard deviation over five seeds. "
    r"\textbf{Bold} marks the best and \underline{underlining} marks the next-best; methods "
    r"whose intervals overlap are marked together. $\dagger$ denotes methods trained on "
    r"leaked data overlapping with the test set; these are excluded from marking as they "
    r"are not comparable to leakage-controlled methods. N/A denotes unavailable metrics, "
    r"and \textbf{Pre.} denotes whether models use pre-training on external data."
)
CASF_GROUP_HEADERS = ("Core, train-overlap (N=214)", "Non-train (N=124)", "Clean held-out (N=92)")


def _casf_groups(table: Tag):
    """CASF 표의 metric 열 index와 cohort 헤더를 HTML 헤더에서 유도 (모든 cohort 포함).
    Table 1c의 cohort 수(2 또는 3)가 바뀌어도 자동 적응한다."""
    header = table.find("tr")
    cells = header.find_all(["th", "td"], recursive=False)
    indices, headers, start = [], [], 0
    for cell in cells:
        classes = cell.get("class") or []
        if "col-modality" in classes or "col-method" in classes:
            continue
        span = int(cell.get("colspan") or 1)
        text = cell.get_text(" ", strip=True).replace("\xa0", " ")
        n = re.search(r"N\s*=\s*(\d+)", text)
        core = re.sub(r"\s*N\s*=\s*\d+\s*$", "", text.replace("CASF-2016", "")).strip()
        core = (core[:1].upper() + core[1:]) if core else core
        headers.append(rf"{escape_latex_text(core)} (N={n.group(1)})" if n else escape_latex_text(text))
        indices.extend(range(start, start + span))
        start += span
    return tuple(indices), tuple(headers), start


def casf_to_latex(table: Tag) -> str:
    indices, headers, total = _casf_groups(table)
    return _modality_grouped_to_latex(
        table, indices, headers,
        CASF_CAPTION, "tab:result-binding-affinity-casf", expect_metrics=total,
        casf_clean=frozenset({"IPNet (frozen)"}))


def _similarity_groups(table: Tag):
    """similarity-stratified 표의 metric 열 index와 그룹 헤더를 HTML 헤더에서 유도 (LP-PDBBind 열 제외)."""
    header = table.find("tr")
    cells = header.find_all(["th", "td"], recursive=False)
    indices, headers, start = [], [], 0
    for cell in cells:
        classes = cell.get("class") or []
        if "col-modality" in classes or "col-method" in classes:
            continue
        span = int(cell.get("colspan") or 1)
        text = cell.get_text(" ", strip=True).replace("\xa0", " ")
        if "LP-PDBBind" not in text:            # full-test 열은 Table 1과 중복이라 제외
            thr = re.search(r"<\s*(\d+)\s*%", text)
            n = re.search(r"N\s*=\s*(\d+)", text)
            label = (rf"Sequence identity $<${thr.group(1)}\% (N={n.group(1)})"
                     if thr and n else escape_latex_text(text))
            indices.extend(range(start, start + span))
            headers.append(label)
        start += span
    return tuple(indices), tuple(headers), start


def similarity_stratified_to_latex(table: Tag) -> str:
    indices, headers, total = _similarity_groups(table)
    return _modality_grouped_to_latex(
        table, indices, headers, SIMILARITY_MODALITY_CAPTION,
        "tab:result-binding-affinity-seqsim", expect_metrics=total)


def binding_affinity_cl1_cl2_to_latex(table: Tag) -> str:
    """CL1/CL2-filtered tiers of the binding-affinity table, in the modality (Pre. 열) format."""
    return _modality_grouped_to_latex(
        table, (3, 4, 5, 6, 7, 8),
        ("CL1 filtered (N=1166)", "CL2 filtered (N=1149)"),
        BINDING_CL1_CL2_CAPTION, "tab:result-binding-affinity-cl1-cl2", expect_metrics=12)


# ── CASF-2016 clean-92 focused table: Pearson r with a dedicated 90% CI column ──
# New table type (same pattern as the CL1/CL2 table above). Reads the CASF Table 1c
# (tables[2], non-train + clean), keeps ONLY the clean-92 cohort, and splits the BCa
# 90% CI into its own column. std parsed from the '±' part only (CI is separate).

CASF_ABS = frozenset({"DSMBind"})   # zero-shot |r| — excluded from ranking (matches HTML unranked_ctx)
_NUM = r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)"


def _casf_parse_metric_ci(cell):
    """(mean, std, (lo,hi)|None). std ONLY from the '±...' part (not the CI); CI from '[lo, hi]'."""
    tbd = cell.find(class_="tbd")
    if tbd is not None and tbd.get_text(strip=True).lower() in ("n/a", "na"):
        return (None, None, None)
    val = cell.find(class_="val")
    if val is None:
        return None
    mean = float(re.search(_NUM, val.get_text(strip=True).replace("−", "-")).group())
    std, ci = 0.0, None
    sd = cell.find(class_="sd")
    if sd:
        txt = sd.get_text(" ", strip=True).replace("\xa0", " ")
        sm = re.search(r"±\s*(" + _NUM + ")", txt)
        if sm:
            std = abs(float(sm.group(1)))
        cm = re.search(r"\[\s*(" + _NUM + r")\s*,\s*(" + _NUM + r")\s*\]", txt)
        if cm:
            ci = (float(cm.group(1)), float(cm.group(2)))
    return (mean, std, ci)


def casf_clean_ci_to_latex(table):
    indices, headers, total = _casf_groups(table)
    clean_g = next((gi for gi, h in enumerate(headers) if re.search(r"clean|held", h, re.I)), len(headers) - 1)
    clean_idx = list(indices[3 * clean_g:3 * clean_g + 3])

    html_rows = []
    for row in table.find_all("tr"):
        method_cell = row.find("td", class_="col-method", recursive=False)
        metrics = row.find_all("td", class_="metric", recursive=False)
        if method_cell is None or len(metrics) != total:
            continue
        text = method_cell.get_text(" ", strip=True)
        html_rows.append((text, [_casf_parse_metric_ci(metrics[i]) for i in clean_idx]))

    casf_clean = frozenset({"IPNet (frozen)"})
    resolved = []
    for _, methods in BINDING_MODALITY_GROUPS:
        for display, key, pre, leaked in methods:
            match = next((v for text, v in html_rows if text.startswith(key)), None)
            if match is None:
                raise ValueError(f"clean-CASF: {key!r} 행 없음")
            disp_e, leaked_e = ((display.replace(r"$^\dagger$", ""), False)
                                if key in casf_clean else (display, leaked))
            excluded = leaked_e or key in CASF_ABS      # not eligible for the best-highlight
            resolved.append((disp_e, pre, excluded, match))

    maximize = [True, True, False]
    bold = [set() for _ in range(3)]
    under = [set() for _ in range(3)]
    for col in range(3):
        pts = [(i, r[3][col][0], r[3][col][1]) for i, r in enumerate(resolved)
               if not r[2] and r[3][col] is not None and r[3][col][0] is not None]
        if not pts:
            continue
        best = (max if maximize[col] else min)(m for _, m, _ in pts)
        reaches = ((lambda m, s: m + s >= best - 1e-12) if maximize[col]
                   else (lambda m, s: m - s <= best + 1e-12))
        for i, m, s in pts:
            if reaches(m, s):
                bold[col].add(i)
        rest = [(i, m) for i, m, s in pts if i not in bold[col]]
        if rest:
            sec = (max if maximize[col] else min)(m for _, m in rest)
            for i, m in rest:
                if abs(m - sec) < 1e-9:
                    under[col].add(i)

    def mcell(i, col, tr):
        if tr is None:
            return "TBA"
        m, s, _ = tr
        if m is None:
            return "N/A"
        if not maximize[col] and m >= 100:
            return "N/A"
        t = rf"{m:.3f}\std{{{s:.3f}}}" if s else rf"{m:.3f}"   # hide ±0.000 (single-vector)
        if i in bold[col]:
            return rf"\textbf{{{t}}}"
        if i in under[col]:
            return rf"\underline{{{t}}}"
        return t

    def cicell(tr):
        if tr is None or tr[0] is None or tr[2] is None:
            return r"\textendash"
        lo, hi = tr[2]
        return rf"{{\scriptsize $[{lo:.2f},\,{hi:.2f}]$}}"

    lines = []
    i = 0
    for modality, methods in BINDING_MODALITY_GROUPS:
        lines.append(rf"\multirow{{{len(methods)}}}{{*}}{{{modality}}}")
        for display, key, pre, leaked in methods:
            vals = resolved[i][3]
            cells = [resolved[i][0], pre,
                     mcell(i, 0, vals[0]), cicell(vals[0]),
                     mcell(i, 1, vals[1]), cicell(vals[1]),
                     mcell(i, 2, vals[2]), cicell(vals[2])]
            lines.append("& " + " & ".join(cells) + r" \\")
            i += 1
        if modality != BINDING_MODALITY_GROUPS[-1][0]:
            lines.append(r"\midrule")

    caption = (
        r"\textbf{CASF-2016 clean held-out (N=92) binding-affinity prediction.} "
        r"All models are trained on LP-PDBBind and evaluated on the CASF-2016 core complexes held out "
        r"from both our train and validation splits. Pearson \textit{r} / Spearman $\rho$ / RMSE are the "
        r"5-seed mean\,$\pm$\,std, each followed by its \textbf{90\% CI} --- a BCa bootstrap that resamples the "
        r"test complexes and averages the per-seed metric, matching CASF-2016 \S2.4. "
        r"Methods shown \emph{without} $\pm$std are single deterministic models (no training seed) --- the "
        r"zero-shot scorers Nesso-1 and DSMBind; for these the interval is pure test-set sampling. "
        r"\textbf{Bold} marks results tied with the best (error bar reaches the best mean); "
        r"\underline{underline} marks the runner-up. $^\dagger$: affinity-leaked reference.")
    header1 = (r"\multirow{2}{*}{\textbf{Input}} & \multirow{2}{*}{\textbf{Method}} & "
               r"\multirow{2}{*}{\textbf{Pre.}} & \multicolumn{6}{c}{\textbf{CASF-2016 clean held-out (N\,=\,92)}} \\")
    header2 = (r"& & & \textbf{Pearson \textit{r}} & \textbf{90\% CI} & "
               r"\textbf{Spearman $\rho$} & \textbf{90\% CI} & "
               r"\textbf{RMSE $\downarrow$} & \textbf{90\% CI} \\")
    body = [
        r"\begin{table}[!t]",
        r"    \centering",
        r"    \caption{",
        f"        {caption}",
        r"    }",
        r"    \label{tab:result-casf-clean}",
        r"    \resizebox{.98\textwidth}{!}{%",
        r"        \begin{tabular}{@{}llccccccc@{}}",
        r"            \toprule",
        "            " + header1,
        r"            \cmidrule(lr){4-9}",
        "            " + header2,
        r"            \midrule",
    ]
    body.extend("            " + ln for ln in lines)
    body.extend([
        r"            \bottomrule",
        r"        \end{tabular}",
        r"    }",
        r"\end{table}",
    ])
    return "\n".join(body)

## Load report + list tables

In [ ]:
resolved = resolve_html_path(HTML_PATH)
soup = BeautifulSoup(resolved.read_text(encoding="utf-8"), "html.parser")
tables = soup.find_all("table")
print(f"Source: {resolved}  ·  {len(tables)} tables")
for i, t in enumerate(tables):
    rows, _, _, ncols = build_layout(t)
    print(f"  [{i}] {len(rows)}x{ncols}  {table_title(t, i)}")


## Table 1a — binding affinity (Test + CL3)

In [ ]:
print(binding_affinity_to_latex(tables[0]))

## CL1 / CL2 filtered tiers

In [ ]:
print(binding_affinity_cl1_cl2_to_latex(tables[0]))

## Similarity-stratified (protein id < 60% / < 30%)

In [ ]:
print(similarity_stratified_to_latex(tables[1]))

## CASF-2016 (core / non-train / clean)

In [ ]:
print(casf_to_latex(tables[2]))

## CASF clean held-out (with 90% CI)

In [ ]:
print(casf_clean_ci_to_latex(tables[2]))